# AIインフラ・ウォッチリスト スクリーニング

作成: 2026-06-28 / しぐまAI

目的: [[モデルポートフォリオ案]]・[[運用ダッシュボード]]を数字で裏付ける。AIインフラ全層の主要銘柄を、
**質（利益率・モート）× 押し目（52週高値からの距離）× バリュエーション（予想PER）× 相関（集中リスク）**でスクリーニングし、
「質が高く、押していて、混んでいない」候補を機械的に surface する。

週次で再実行してダッシュボードを更新する想定。カーネルは `investment-notes`。

注意: 予想PERが低い＝割安とは限らない（メモリ等シクリカルはピーク益で低PERになる＝割安の罠）。質×押し目スコアと合わせて読む。

In [ ]:
import yfinance as yf
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
fx = yf.Ticker("USDJPY=X").history(period="1d")["Close"].iloc[-1]
print("USDJPY =", round(fx, 2))

In [ ]:
# AIインフラ・ウォッチリスト（theme, tier付き）
universe = {
    "NVDA": ("NVIDIA", "compute", "core"),
    "ASML": ("ASML", "compute", "core"),
    "TSM": ("TSMC", "compute", "core"),
    "MU": ("Micron", "memory", "core"),
    "CEG": ("Constellation", "power", "core"),
    "GEV": ("GE Vernova", "power", "core"),
    "ETN": ("Eaton", "power", "core"),
    "VST": ("Vistra", "power", "alpha"),
    "TLN": ("Talen", "power", "alpha"),
    "VRT": ("Vertiv", "cooling", "core"),
    "MOD": ("Modine", "cooling", "alpha"),
    "AVGO": ("Broadcom", "network", "core"),
    "COHR": ("Coherent", "network", "alpha"),
    "LITE": ("Lumentum", "network", "alpha"),
    "MRVL": ("Marvell", "network", "risk"),
    "AMKR": ("Amkor", "packaging", "alpha"),
    "6981.T": ("Murata", "mlcc", "core"),
    "6976.T": ("Taiyo Yuden", "mlcc", "alpha"),
}

In [ ]:
def momentum(c):
    last = float(c.iloc[-1])
    off_high = (last / float(c.max()) - 1) * 100
    m1 = (last / float(c.iloc[-22]) - 1) * 100 if len(c) > 22 else None
    m3 = (last / float(c.iloc[-66]) - 1) * 100 if len(c) > 66 else None
    return last, off_high, m1, m3

rows = []
for tk, (label, theme, tier) in universe.items():
    t = yf.Ticker(tk)
    c = t.history(period="1y", auto_adjust=False)["Close"].dropna()
    if len(c) < 2:
        continue
    last, off_high, m1, m3 = momentum(c)
    info = t.info
    mc = info.get("marketCap")
    mc_usd = (mc / fx if tk.endswith(".T") else mc) / 1e9 if mc else None
    margin = info.get("profitMargins")
    rows.append({
        "ticker": tk, "label": label, "theme": theme, "tier": tier,
        "off_52wH%": round(off_high, 1),
        "1m%": round(m1, 1) if m1 else None,
        "3m%": round(m3, 1) if m3 else None,
        "fwdPE": round(info.get("forwardPE"), 1) if info.get("forwardPE") else None,
        "margin%": round(margin * 100, 1) if margin else None,
        "mc$B": round(mc_usd, 1) if mc_usd else None,
    })
df = pd.DataFrame(rows)
df

## 「質 × 押し目」スクリーン

`quality_dip = margin% × (52週高値からの下落率) / 100`。利益率（質・モートの代理）が高く、かつ大きく押している銘柄ほど高スコア。
**シクリカル（メモリ）はピーク益で低PER・高marginになりがち→低PERを割安と即断しない。**

In [ ]:
df["quality_dip"] = (df["margin%"].fillna(0)) * (-df["off_52wH%"]) / 100
df.sort_values("quality_dip", ascending=False).reset_index(drop=True)

## 相関（集中リスク）

相関0.7超は実質1銘柄。合計サイズで損失管理する（[[投資ルール]]）。

In [ ]:
core = ["NVDA", "AVGO", "MU", "CEG", "GEV", "VRT", "COHR", "LITE", "ASML", "TSM"]
px = yf.download(core, period="3mo", interval="1d", progress=False)["Close"]
px.pct_change().dropna().corr().round(2)

## 読み方メモ（2026-06-28時点の所見）

- **NVDA: 予想PER ~15 × 利益率63% × -18%押し** = 質の押し目筆頭。AI計算の独占モートが市場並み倍率。
- **AVGO: -24%押し × 利益率39% × PER19** = 2027見通し据え置きの失望で過剰反応の可能性。
- **CEG: -35%押し × PER20** = AI電力の本命が深く押す（実キャッシュフロー、[[AI電力_原子力・SMR・ガス]]）。
- **MU: PER7.6 × margin56%だが-6.7%しか押していない** = ピーク益の低PER。割安の罠に注意（シクリカル）。
- **TLN: margin約ゼロ** = 電力でも財務の質は劣る。CEG優位。
- 相関: フォトニクス(COHR/LITE)とAI半導体は高相関。電力(CEG/GEV)は相対的に非相関＝分散効果（[[AIインフラ_どこに賭けるか]]）。
- 関連: [[モデルポートフォリオ案]] / [[運用ダッシュボード]] / [[AIトレードのプレモータム]]。